# Setting up a Simulation

First, we import the modules we need:

In [6]:
from ase.io import read
from ase.visualize import view

### Choosing the Starting structure

Now we load the structure that we previously created:

In [7]:
atoms = read("03_structure.cif")

Quick check if the structure looks correct:

In [8]:
view(atoms)

<Popen: returncode: None args: ['/scratch/data/marion/miniforge3/envs/ml/bin...>

### Choosing the Model

Now that we got our initial Structure we can choose a model that suits our use case. There are different models available.

Pick a machine‑learned potential suited to your system:

- Universal MOFs and general solids: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mp_0b (MACE‑MP‑0b, different models, recommended: 
mace_agnesi_small.model)
- Newest MACE model for materials: https://github.com/ACEsuit/mace-foundations (MACE-mh-1, head: omat_pbe)
- MOF‑specific: https://github.com/ddmms/data/tree/main/mace-mof-0/v2 (MACE‑MP‑MOFv2, head: pbe_d3)
- Small molecules: github.com/ACEsuit/mace-off (MACE‑OFF; not suited for MOFs)
- Alternative approach:  (UMA different methodology)


General: MACE Foundations (models and docs): https://github.com/ACEsuit/mace-foundations

For this workshop we want to use a MACE model that suits MOFs (mace_agnesi_small.model, mace-mh-1.model or mofs_v2.model).

In [9]:
from mace.calculators import mace

/scratch/data/marion/miniforge3/envs/ml/lib/python3.13/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


Now we include the model. Adjust the path to where you saved the model you downloaded

The models are float64 by default but we use float32 to conserve computational effort.

When we calculate our trajectories we use CUDA but for now to check we can simply use the cpu.

In [16]:
model = "../../models/mace-mh-1.model"
default_dtype = "float32"
calc = mace.MACECalculator(model_paths=model, default_dtype=default_dtype ,device="cpu", head="omat_pbe")

/scratch/data/marion/miniforge3/envs/ml/lib/python3.13/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [25]:
atoms.calc = calc

# Test energy/forces
e = atoms.get_potential_energy()
f = atoms.get_forces()
print("Initial energy (eV):", e)
print("Max force (eV/Å):", abs(f).max())

Initial energy (eV): -866.5420532226562
Max force (eV/Å): 0.044957936


### Minimizing the system

In [26]:
from ase.optimize import FIRE

In [27]:
opt = FIRE(atoms, logfile=None)
opt.run(fmax=0.05)  # relax until max force <= 0.05 eV/Å

np.True_

In [23]:
view(atoms)

<Popen: returncode: None args: ['/scratch/data/marion/miniforge3/envs/ml/bin...>

### Equilibrate the system

In [28]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.io.trajectory import Trajectory
from ase import units
import numpy as np
import os

In [30]:
# 1) Initialize velocities and remove net momentum
MaxwellBoltzmannDistribution(atoms, temperature_K=300.0, rng=np.random.default_rng(42))
Stationary(atoms)

dt = 0.25 * units.fs
fric = 0.05
dyn = Langevin(atoms, dt, temperature_K=300.0, friction=fric)

report_interval = 100

steps = 1000

# 2) Clean observers, attach logger once
dyn.observers = []
os.makedirs("traj", exist_ok=True)
os.makedirs("out", exist_ok=True)

with Trajectory("traj/03_equilibration.traj", "w", atoms) as traj:
    dyn.attach(traj.write, interval=report_interval)

    def print_status():
        ekin = atoms.get_kinetic_energy()
        epot = atoms.get_potential_energy()
        N = len(atoms)
        T = 2.0 * ekin / (3.0 * N * units.kB)  # adjust ndof if you have constraints
        print(f"step={dyn.get_number_of_steps():4d}  Epot={epot:10.3f} eV  "
              f"Ekin={ekin:10.3f} eV  T={T:7.1f} K")
    dyn.attach(print_status, interval=report_interval)

    dyn.run(steps)

/scratch/data/marion/miniforge3/envs/ml/lib/python3.13/site-packages/ase/md/langevin.py:110: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


step=   0  Epot=  -866.542 eV  Ekin=     4.092 eV  T=  270.6 K


KeyboardInterrupt: 